# 6 — Predictions and illustrative review routing

Use the saved numeric GLM predictions from Notebook 2 for all 1,039 validation claims.
Compare all 25 run combinations for each of the six model pairs within the
1%-of-baseline MAE tolerance. Summaries average comparison-specific metrics,
not predictions. The original randomized labels include rerun and context variation.

Inputs: `intermediate/s2_analysis.pkl`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from itertools import combinations, product
from scipy.stats import spearmanr
from IPython.display import Markdown

In [2]:
INTERMEDIATE = Path("intermediate")
numeric = pd.read_pickle(INTERMEDIATE / "s2_analysis.pkl")
claims = numeric["validation_claims"]
MODEL_ORDER = numeric["model_order"]
RUNS = numeric["runs"]
validation_predictions = numeric["validation_predictions"]
baseline_mae = numeric["baseline_mae"]

In [3]:
claims = (
    claims.loc[claims.split.eq("validation")]
    .sort_values("claim_row")
    .reset_index(drop=True)
)
N = len(claims)
REVIEW_N = int(np.ceil(N / 10))
TOTAL_VALIDATION_LOSS = claims.Loss.sum()
predictions = np.stack(
    [
        np.column_stack(
            [
                validation_predictions.loc[
                    validation_predictions.model.eq(model)
                    & validation_predictions.run.eq(run)
                ]
                .sort_values("claim_row")
                .prediction.to_numpy()
                for run in RUNS
            ]
        )
        for model in MODEL_ORDER
    ],
    axis=1,
)
mean_mae = (
    numeric["performance_summary"]
    .set_index("model")
    .loc[MODEL_ORDER, "mean_mae"]
    .to_numpy()
)
eligible_pairs = [
    (a, b)
    for a, b in combinations(range(len(MODEL_ORDER)), 2)
    if abs(mean_mae[a] - mean_mae[b]) < 0.01 * baseline_mae
]

## Deciles and review membership

Sort by descending prediction, then ascending original claim-row order for ties.
Decile 10 contains 104 claims. A switch means membership in exactly one review
set, counting both directions. A second policy includes all claims tied at the
cutoff, allowing variable capacity. Predictions remain in `log1p(Loss)` units.

In [4]:
deciles = np.empty(predictions.shape, dtype=int)
review = np.zeros(predictions.shape, dtype=bool)
inclusive_review = np.zeros(predictions.shape, dtype=bool)
boundary_rows = []
for m, model in enumerate(MODEL_ORDER):
    for r, run in enumerate(RUNS):
        score = predictions[:, m, r]
        order = np.lexsort((claims.claim_row.to_numpy(), -score))
        deciles[order, m, r] = 10 - (10 * np.arange(N) // N)
        review[:, m, r] = deciles[:, m, r] == 10
        cutoff = float(score[order[REVIEW_N - 1]])
        inclusive_review[:, m, r] = score >= cutoff
        above = int((score > cutoff).sum())
        tied = int((score == cutoff).sum())
        boundary_rows.append(
            {
                "model": model,
                "run": run,
                "cutoff_score": cutoff,
                "claims_above_cutoff": above,
                "claims_tied_at_cutoff": tied,
                "tied_claims_selected": REVIEW_N - above,
                "capacity_tie_break_used": REVIEW_N - above < tied,
                "fixed_review_count": REVIEW_N,
                "inclusive_review_count": int(inclusive_review[:, m, r].sum()),
            }
        )
cutoff_details = pd.DataFrame(boundary_rows)
print(
    f"Fixed-capacity rule selects {REVIEW_N} claims per model/run ({REVIEW_N / N:.2%})."
)
print(
    f"{cutoff_details.capacity_tie_break_used.sum()}/30 fits require a tie-break at the review cutoff."
)
print(
    "With all cutoff ties included, review counts range from "
    f"{cutoff_details.inclusive_review_count.min()} to {cutoff_details.inclusive_review_count.max()} claims."
)

Fixed-capacity rule selects 104 claims per model/run (10.01%).
29/30 fits require a tie-break at the review cutoff.
With all cutoff ties included, review counts range from 104 to 132 claims.


## Prediction differences after GLM recalibration

In [5]:
pair_rows = []
loss = claims.Loss.to_numpy()
for a, b in eligible_pairs:
    for ra, rb in product(range(5), repeat=2):
        pa, pb = predictions[:, a, ra], predictions[:, b, rb]
        route_a, route_b = review[:, a, ra], review[:, b, rb]
        switch = route_a != route_b
        inclusive_switch = inclusive_review[:, a, ra] != inclusive_review[:, b, rb]
        entering = int((~route_a & route_b).sum())
        leaving = int((route_a & ~route_b).sum())
        pair_rows.append(
            {
                "model_a": MODEL_ORDER[a],
                "model_b": MODEL_ORDER[b],
                "run_a": RUNS[ra],
                "run_b": RUNS[rb],
                "mean_abs_prediction_difference": float(np.abs(pa - pb).mean()),
                "prediction_spearman": float(spearmanr(pa, pb).statistic),
                "different_decile_pct": float(
                    100 * np.mean(deciles[:, a, ra] != deciles[:, b, rb])
                ),
                "highest_decile_change_pct": float(100 * switch.mean()),
                "routing_switch_count": int(switch.sum()),
                "entering_review": entering,
                "leaving_review": leaving,
                "switching_claim_realized_loss": float(loss[switch].sum()),
                "switching_loss_pct_validation": float(
                    100 * loss[switch].sum() / TOTAL_VALIDATION_LOSS
                ),
                "inclusive_switch_count": int(inclusive_switch.sum()),
                "inclusive_switch_realized_loss": float(loss[inclusive_switch].sum()),
                "inclusive_review_count_a": int(inclusive_review[:, a, ra].sum()),
                "inclusive_review_count_b": int(inclusive_review[:, b, rb].sum()),
            }
        )
run_pair_results = pd.DataFrame(pair_rows)
summary = (
    run_pair_results.groupby(["model_a", "model_b"], sort=False)
    .agg(
        mean_abs_prediction_difference=("mean_abs_prediction_difference", "mean"),
        prediction_spearman=("prediction_spearman", "mean"),
        different_decile_pct=("different_decile_pct", "mean"),
        highest_decile_change_pct=("highest_decile_change_pct", "mean"),
        mean_routing_switches=("routing_switch_count", "mean"),
        min_routing_switches=("routing_switch_count", "min"),
        max_routing_switches=("routing_switch_count", "max"),
        mean_switching_realized_loss=("switching_claim_realized_loss", "mean"),
        min_switching_realized_loss=("switching_claim_realized_loss", "min"),
        max_switching_realized_loss=("switching_claim_realized_loss", "max"),
        mean_switching_loss_pct_validation=("switching_loss_pct_validation", "mean"),
        inclusive_mean_switches=("inclusive_switch_count", "mean"),
        inclusive_min_switches=("inclusive_switch_count", "min"),
        inclusive_max_switches=("inclusive_switch_count", "max"),
        inclusive_mean_switching_realized_loss=(
            "inclusive_switch_realized_loss",
            "mean",
        ),
    )
    .reset_index()
)
summary["mae_gap_pct_baseline"] = [
    100
    * abs(mean_mae[MODEL_ORDER.index(a)] - mean_mae[MODEL_ORDER.index(b)])
    / baseline_mae
    for a, b in zip(summary.model_a, summary.model_b)
]
summary["LLM pair"] = summary.model_a + " / " + summary.model_b

comparison_table = summary[
    [
        "LLM pair",
        "mae_gap_pct_baseline",
        "mean_abs_prediction_difference",
        "prediction_spearman",
        "different_decile_pct",
        "highest_decile_change_pct",
    ]
].rename(
    columns={
        "mae_gap_pct_baseline": "MAE gap (% baseline)",
        "mean_abs_prediction_difference": "Mean absolute prediction difference",
        "prediction_spearman": "Prediction Spearman rho",
        "different_decile_pct": "Different decile (%)",
        "highest_decile_change_pct": "Highest-decile membership change (%)",
    }
)
display(
    comparison_table.style.hide(axis="index").format(
        {
            "MAE gap (% baseline)": "{:.3f}",
            "Mean absolute prediction difference": "{:.4f}",
            "Prediction Spearman rho": "{:.4f}",
            "Different decile (%)": "{:.2f}",
            "Highest-decile membership change (%)": "{:.2f}",
        }
    )
)

LLM pair,MAE gap (% baseline),Mean absolute prediction difference,Prediction Spearman rho,Different decile (%),Highest-decile membership change (%)
GPT-5.6 Sol / GLM-5.2,0.934,0.2477,0.9269,39.98,5.47
Kimi K2.6 / DeepSeek V4 Pro,0.119,0.2601,0.9275,47.74,6.81
Kimi K2.6 / Qwen3.5 397B,0.987,0.2334,0.9349,46.67,5.39
Kimi K2.6 / GLM-5.2,0.296,0.2827,0.9286,52.25,7.28
DeepSeek V4 Pro / Qwen3.5 397B,0.868,0.2716,0.9329,55.40,7.20
DeepSeek V4 Pro / GLM-5.2,0.415,0.2730,0.9251,50.83,6.54


## Switching claims and their realized losses

Loss totals sum observed paid losses once per switching claim within a comparison,
then average across comparisons. They are not causal costs, benefits, or observed
customer harm, and must not be added across overlapping comparisons.

In [6]:
triage_table = pd.DataFrame(
    {
        "LLM pair": summary["LLM pair"],
        "Mean switching claims": summary.mean_routing_switches,
        "Switching claims range": [
            f"{a}–{b}"
            for a, b in zip(summary.min_routing_switches, summary.max_routing_switches)
        ],
        "Mean realized loss of switching claims": summary.mean_switching_realized_loss,
        "Realized loss range": [
            f"${a:,.0f}–${b:,.0f}"
            for a, b in zip(
                summary.min_switching_realized_loss, summary.max_switching_realized_loss
            )
        ],
        "Mean share of validation loss (%)": summary.mean_switching_loss_pct_validation,
    }
)
display(
    triage_table.style.hide(axis="index").format(
        {
            "Mean switching claims": "{:.1f}",
            "Mean realized loss of switching claims": "${:,.0f}",
            "Mean share of validation loss (%)": "{:.2f}",
        }
    )
)

LLM pair,Mean switching claims,Switching claims range,Mean realized loss of switching claims,Realized loss range,Mean share of validation loss (%)
GPT-5.6 Sol / GLM-5.2,56.8,36–70,"$3,955,319","$1,086,259–$6,085,249",17.72
Kimi K2.6 / DeepSeek V4 Pro,70.8,48–86,"$4,411,853","$2,011,054–$6,584,460",19.76
Kimi K2.6 / Qwen3.5 397B,56.0,38–70,"$3,789,714","$937,215–$5,920,376",16.97
Kimi K2.6 / GLM-5.2,75.7,60–84,"$4,836,819","$2,022,300–$7,203,927",21.66
DeepSeek V4 Pro / Qwen3.5 397B,74.8,50–96,"$4,251,530","$2,120,220–$6,422,797",19.04
DeepSeek V4 Pro / GLM-5.2,67.9,48–82,"$4,078,413","$1,995,858–$6,663,839",18.27


## Sensitivity to cutoff ties

In [7]:
tie_summary = (
    cutoff_details.groupby("model", sort=False)
    .agg(
        fits_requiring_tie_break=("capacity_tie_break_used", "sum"),
        min_cutoff_tie_size=("claims_tied_at_cutoff", "min"),
        max_cutoff_tie_size=("claims_tied_at_cutoff", "max"),
        min_inclusive_reviews=("inclusive_review_count", "min"),
        max_inclusive_reviews=("inclusive_review_count", "max"),
    )
    .reset_index()
)
display(tie_summary)
sensitivity_table = pd.DataFrame(
    {
        "LLM pair": summary["LLM pair"],
        "Fixed-capacity mean switches": summary.mean_routing_switches,
        "Include-ties mean switches": summary.inclusive_mean_switches,
        "Include-ties switch range": [
            f"{a}–{b}"
            for a, b in zip(
                summary.inclusive_min_switches, summary.inclusive_max_switches
            )
        ],
        "Include-ties mean realized loss of switching claims": summary.inclusive_mean_switching_realized_loss,
    }
)
display(
    sensitivity_table.style.hide(axis="index").format(
        {
            "Fixed-capacity mean switches": "{:.1f}",
            "Include-ties mean switches": "{:.1f}",
            "Include-ties mean realized loss of switching claims": "${:,.0f}",
        }
    )
)

,model,fits_requiring_tie_break,min_cutoff_tie_size,max_cutoff_tie_size,min_inclusive_reviews,max_inclusive_reviews
0,GPT-5.6 Sol,5,18,29,108,115
1,Gemini 3.1 Pro,5,13,21,106,124
2,Kimi K2.6,5,8,29,106,115
3,DeepSeek V4 Pro,5,26,31,114,132
4,Qwen3.5 397B,4,6,19,104,119
5,GLM-5.2,5,18,27,105,121


LLM pair,Fixed-capacity mean switches,Include-ties mean switches,Include-ties switch range,Include-ties mean realized loss of switching claims
GPT-5.6 Sol / GLM-5.2,56.8,61.7,37–86,"$2,659,864"
Kimi K2.6 / DeepSeek V4 Pro,70.8,64.2,45–82,"$3,893,916"
Kimi K2.6 / Qwen3.5 397B,56.0,63.6,47–76,"$3,900,651"
Kimi K2.6 / GLM-5.2,75.7,88.2,72–104,"$5,048,272"
DeepSeek V4 Pro / Qwen3.5 397B,74.8,67.6,51–88,"$3,688,095"
DeepSeek V4 Pro / GLM-5.2,67.9,84.0,61–103,"$4,388,763"


In [8]:
report = (
    f"**Existing GLMs:** all 31,170 validation predictions are loaded from the numeric fits in Notebook 2. "
    f"Across the six eligible LLM pairs, the mean absolute prediction difference ranges from "
    f"**{summary.mean_abs_prediction_difference.min():.4f} to {summary.mean_abs_prediction_difference.max():.4f}** "
    f"in predicted `log1p(Loss)`, with mean prediction Spearman correlation "
    f"**{summary.prediction_spearman.min():.4f}–{summary.prediction_spearman.max():.4f}**. "
    f"Mean different-decile rates are **{summary.different_decile_pct.min():.2f}%–{summary.different_decile_pct.max():.2f}%**.\n\n"
    f"**Illustrative review rule (104 claims per run):** pair-level mean routing changes range from "
    f"**{summary.mean_routing_switches.min():.1f} to {summary.mean_routing_switches.max():.1f} claims** "
    f"(**{summary.highest_decile_change_pct.min():.2f}%–{summary.highest_decile_change_pct.max():.2f}%** of validation claims). "
    f"Across individual run comparisons, actual switch counts range from "
    f"{run_pair_results.routing_switch_count.min()} to {run_pair_results.routing_switch_count.max()}. "
    f"The pair-level mean realized paid loss associated with switching claims ranges from "
    f"**${summary.mean_switching_realized_loss.min():,.0f} to ${summary.mean_switching_realized_loss.max():,.0f}**.\n\n"
    f"**Cutoff ties:** {cutoff_details.capacity_tie_break_used.sum()}/30 fits require a tie-break. "
    f"Including all cutoff ties gives {cutoff_details.inclusive_review_count.min()}–"
    f"{cutoff_details.inclusive_review_count.max()} reviews per run and "
    f"{summary.inclusive_mean_switches.min():.1f}–{summary.inclusive_mean_switches.max():.1f} mean routing switches across pairs.\n\n"
    "Similar aggregate MAE remains compatible with different fitted predictions and illustrative routing after "
    "coefficient recalibration. These are potential differences under an illustrative rule, "
    "not measured economic costs or benefits, actual claim decisions, or proof of materiality in deployment."
)
display(Markdown(report.replace("\\n", "\n")))

**Existing GLMs:** all 31,170 validation predictions are loaded from the numeric fits in Notebook 2. Across the six eligible LLM pairs, the mean absolute prediction difference ranges from **0.2334 to 0.2827** in predicted `log1p(Loss)`, with mean prediction Spearman correlation **0.9251–0.9349**. Mean different-decile rates are **39.98%–55.40%**.

**Illustrative review rule (104 claims per run):** pair-level mean routing changes range from **56.0 to 75.7 claims** (**5.39%–7.28%** of validation claims). Across individual run comparisons, actual switch counts range from 36 to 96. The pair-level mean realized paid loss associated with switching claims ranges from **$3,789,714 to $4,836,819**.

**Cutoff ties:** 29/30 fits require a tie-break. Including all cutoff ties gives 104–132 reviews per run and 61.7–88.2 mean routing switches across pairs.

Similar aggregate MAE remains compatible with different fitted predictions and illustrative routing after coefficient recalibration. These are potential differences under an illustrative rule, not measured economic costs or benefits, actual claim decisions, or proof of materiality in deployment.

The review rule is hypothetical. Claim-description timing, actual handling
decisions, intervention effects, and service outcomes are unobserved.

## Save results

In [9]:
pd.to_pickle(
    {
        "run_pair_results": run_pair_results,
        "pair_summary": summary,
        "cutoff_details": cutoff_details,
        "tie_summary": tie_summary,
    },
    INTERMEDIATE / "s6_routing.pkl",
)